# Programming Assignment 4: Dimensionality Reduction
**Twitter Entity Sentiment - feature selection metrics, LSA with SVD, and t-SNE**

## 📋 Assignment Information

**Course:** Unstructured Data Analysis (2026-2)
**Assignment:** Programming Assignment 4 - Dimensionality Reduction
**Released:** October 19, 2026 (Mon), 09:00
**Deadline:** October 25, 2026 (Sun), 23:59
**Total:** 100 points

## 🎯 Objectives

Assignments 2 and 3 built term-document matrices and dense vectors of the tweets. This assignment makes those matrices
**smaller**, with the two roads of lecture 6-2 / 7-1 and the Week 7 notebooks:

- **Feature selection**: implement the lecture's **ten metrics** (DF, Acc, AccR, PR, OddR, OddN, F1, IG, $\chi^2$, BNS)
  on a real binary term-document matrix and check them on the lecture's 10 x 10 example
- **Feature extraction**: decompose a term-document matrix with NumPy's **SVD** (rank, truncation, error), then run **LSA**
  with `TruncatedSVD` on the TF-IDF matrix and read what a latent dimension means
- **SNE**: compute the neighbor probabilities $p_{j|i}$ and find the radius $\sigma_i$ that gives a chosen **perplexity**

This is a short assignment (it falls in the week before the midterm): five fill-in-the-blank TODO items, 30 blanks, nothing else.

Everything is inside lecture 6-2 / 7-1 and the two Week 7 notebooks. No classifier is involved.

The data and settings differ from the Week 7 notebooks on purpose (tweets instead of movie reviews, 50 latent
dimensions): the methods are the same, the numbers are not, so do not compare your outputs with the class notebooks.

**Your parameters are assigned from your student ID.** Section 0 asks for your ID and derives from it the words, the
entity, the dimension, the point and the perplexity you work on (`MY`). The grader recomputes them from your ID, so the
outputs it expects are yours, not a classmate's.

## 📝 Instructions

1. `File ▸ Save a copy in Drive`, then rename the notebook to **`Assignment_4_YourName_StudentID.ipynb`** (e.g. `Assignment_4_HongGilDong_2026123456.ipynb`).
2. Fill in every `________` blank. A blank may be a value, an expression or a whole right-hand side.
   **Do not** change the code outside the blanks and **do not** rename variables: the grader re-runs your notebook.
3. `Runtime ▸ Restart and run all` must finish without an error. **Keep the outputs** in the notebook when you submit.
4. Submit the `.ipynb` file to **"Programming Assignment 4 – Dimensionality Reduction"** under *Assignments*.

**Fill-in-the-blank only.** There are no discussion or multiple-choice questions. All 100 points are for code: the grader
re-runs your notebook and compares **its** outputs with the reference values computed for **your** student ID.

> ⚠️ **Read each specification to the letter.** Several TODOs deliberately ask for something that is *not* the default
> behavior of a library, nor the most common version you will find online or get from a chatbot
> (for example: which logarithm, what replaces a zero count, which SVD solver, how a search stops).
> Only some expected outputs are printed below each TODO, as **checks**; the rest are known only to the grader.
> A blank that "looks right" but ignores the specification produces different numbers and loses its points.

| Part | Content | Points |
|---|---|---|
| 1 | Feature selection: metrics 1-7, metrics 8-10 (TODO 1-2) | 52 |
| 2 | LSA: SVD with NumPy, `TruncatedSVD` (TODO 3-4) | 32 |
| 3 | SNE: $p_{j\|i}$, perplexity and $\sigma_i$ (TODO 5) | 16 |
| | **Total** | **100** |

## 📂 Dataset

The **same** Twitter Entity Sentiment file as Assignments 1-3. Section 0 repeats the Assignment 1 cleaning, keeps the
**Positive** and **Negative** tweets only (the lecture's metrics are defined for two classes), and gives you:

| variable | content |
|---|---|
| `df` | the tweets (`Entity`, `Sentiment`, `Review`, `tokens` without stop words, `text` = the tokens joined) |
| `X_tfidf` | the TF-IDF matrix of `text` (`min_df=5`), used in Part 2 |

The binary term-document matrix `X` and the four counts are given at the start of Part 1.

## &nbsp;0. Setup (given)
*No TODO in this section* except your student ID. Run it once and check the printed numbers:
`documents: 7229`, `Positive: 3472`, `Negative: 3757`, `TF-IDF: (7229, 2188)`.

In [ ]:
import nltk
nltk.download('stopwords', quiet=True)

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)
np.set_printoptions(linewidth=140, suppress=True)

In [ ]:
# --- the Assignment 1 pipeline, given ---------------------------------------
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week03-text-preprocessing/data/twcs.csv"

data = pd.read_csv(DATA_URL)
data_unique = data.drop_duplicates(subset='idx', keep='first')
df = data_unique[data_unique['Sentiment'].isin(['Positive', 'Negative'])].reset_index(drop=True)


def clean_text(d):
    d = str(d).lower()
    d = re.sub(r"http\S+|www\S+|\S+\.com\S*", " ", d)
    d = re.sub(r"[^a-z'\s]", " ", d)
    return d


CUSTOM_STOPS = ['im', 'ive', 'dont', 'cant', 'thats', 'like', 'get', 'got', 'one',
                'game', 'games', 'amp']
stop_words = set(stopwords.words('english')) | set(CUSTOM_STOPS)
tokenizer = RegexpTokenizer(r"[a-z']{2,}")

df['tokens'] = df['Review'].apply(clean_text).apply(tokenizer.tokenize)
df['tokens'] = df['tokens'].apply(lambda ts: [t for t in ts if t not in stop_words])
df['text'] = df['tokens'].apply(' '.join)

tfidf = TfidfVectorizer(min_df=5, token_pattern=r"[a-z']{2,}")
X_tfidf = tfidf.fit_transform(df['text'])

print('documents :', len(df))
print('Positive  :', (df['Sentiment'] == 'Positive').sum())
print('Negative  :', (df['Sentiment'] == 'Negative').sum())
print('TF-IDF    :', X_tfidf.shape)

### Your student ID
Type **your own** 10-digit student ID as a string. Everything personal in this assignment is derived from it.
A wrong ID gives you someone else's questions: your hidden values then match nobody.

In [ ]:
STUDENT_ID = '________'      # <- replace with YOUR student ID

assert STUDENT_ID.isdigit() and len(STUDENT_ID) == 10 and STUDENT_ID != '2026000000', 'Type your own 10-digit student ID'

In [ ]:
# --- your personal parameters (given, do not edit) ---------------------------
# The words, entity, dimension, point and perplexity are assigned
# deterministically from STUDENT_ID. The grader recomputes them from your ID,
# so the ID must be yours.
import hashlib


def _pick(options, salt, k=1):
    order = sorted(range(len(options)),
                   key=lambda n: hashlib.sha256(f'{STUDENT_ID.strip()}|{salt}|{n}'.encode()).hexdigest())
    return [options[n] for n in order[:k]]


_POS = ['love', 'best', 'fun', 'excited', 'awesome', 'great', 'happy']
_NEG = ['fix', 'servers', 'wtf', 'worst', 'hate', 'broken', 'trash', 'sucks']
_NEU = ['new', 'today', 'playing', 'people', 'time', 'day', 'year', 'video']
_ENTITIES = ['MaddenNFL', 'NBA2K', 'AssassinsCreed', 'HomeDepot', 'FIFA', 'TomClancysRainbowSix', 'Verizon',
             'TomClancysGhostRecon', 'Borderlands', 'CallOfDutyBlackopsColdWar', 'PlayStation5(PS5)', 'Microsoft',
             'Overwatch', 'Hearthstone', 'Cyberpunk2077', 'Dota2']
_KSVD = [2, 3, 4]
_DIMS = [2, 3, 4, 5, 6]
_POINTS = [3259, 426, 6529, 6467, 6544, 5946, 4957, 5759, 353, 2932, 642, 6388]
_PERPS = [10, 20, 30, 40, 50]

MY = {
    'words': _pick(_POS, 'pos') + _pick(_NEG, 'neg') + _pick(_NEU, 'neu'),   # TODO 1-2
    'entity': _pick(_ENTITIES, 'entity')[0],                               # TODO 3
    'k_svd': _pick(_KSVD, 'ksvd')[0],                                      # TODO 3
    'dim': _pick(_DIMS, 'dim')[0],                                         # TODO 4
    'point': _pick(_POINTS, 'point')[0],                                   # TODO 5
    'perp': _pick(_PERPS, 'perp')[0],                                      # TODO 5
}
for _k, _v in MY.items():
    print(f'{_k:<7}: {_v}')

## &nbsp;1. Feature Selection (52 pts)

### The binary term-document matrix and the four counts (given)

The matrix the lecture's metrics are defined on: one row per tweet, one column per word, **1 if the word occurs
in the tweet**, words in at least 5 tweets. Then the four counts every metric is made of, **one value per word**:
$N(Pos)$, $N(Neg)$, $N(Pos, w)$, $N(Neg, w)$. This is the Notebook 1 code; *no blank here*, just run it.

**Check:** `X: (7229, 2188)  N(Pos) = 3472.0  N(Neg) = 3757.0` and `fix  N(Pos,w) = 8  N(Neg,w) = 238`.

In [ ]:
# given: X (0/1), and the counts N(Pos), N(Neg), N(Pos, w), N(Neg, w)

vec = CountVectorizer(binary=True, min_df=5, token_pattern=r"[a-z']{2,}")
X = vec.fit_transform(df['text'])                    # documents x terms, entries 0 / 1
terms = vec.get_feature_names_out()
y = (df['Sentiment'] == 'Positive').astype(int).to_numpy()      # 1 = Positive, 0 = Negative  (given)

N_pos = float((y == 1).sum())
N_neg = float((y == 0).sum())
N_pos_w = np.asarray(X[y == 1].sum(axis=0)).ravel().astype(float)     # one value per term
N_neg_w = np.asarray(X[y == 0].sum(axis=0)).ravel().astype(float)

C = dict(N_pos=N_pos, N_neg=N_neg, N_pos_w=N_pos_w, N_neg_w=N_neg_w)
print('X:', X.shape, ' N(Pos) =', N_pos, ' N(Neg) =', N_neg)
for w in ['fix'] + MY['words']:
    i = vec.vocabulary_[w]
    print(f'{w:<10} N(Pos,w) = {N_pos_w[i]:4.0f}   N(Neg,w) = {N_neg_w[i]:4.0f}')

### TODO 1: Metrics 1-7 (24 points)

Complete `metrics_1_to_7(C)`, which returns a DataFrame with the seven columns `DF, Acc, AccR, PR, OddR, OddN, F1`,
computed **for every term at once** from the dictionary of counts `C` (DF and Acc are given), with the lecture's formulas:

$$DF = N(Pos,w) + N(Neg,w) \qquad Acc = N(Pos,w) - N(Neg,w) \qquad
AccR = \left|\frac{N(Pos,w)}{N(Pos)} - \frac{N(Neg,w)}{N(Neg)}\right| \qquad
PR = \frac{N(Pos,w)}{N(Pos)} \bigg/ \frac{N(Neg,w)}{N(Neg)}$$

$$OddR = \frac{N(Pos,w)}{N(Neg,w)} \times \frac{N(Neg,\bar w)}{N(Pos,\bar w)} \qquad OddN = N(Pos,w) \times N(Neg,\bar w) \qquad
F1 = \frac{2 \, N(Pos,w)}{N(Pos) + N(w)}$$

* $PR$ may divide by zero: leave NumPy's `inf`, do not replace it.
* $OddR$: follow the slide, *"add 1 to any zero count in the denominator"*. A count that is **zero** becomes 1;
  a count that is not zero is **not** changed (do not add 1 everywhere, and do not add 1 to the numerators).
* $N(w) = N(Pos,w) + N(Neg,w)$, $N(Pos, \bar w) = N(Pos) - N(Pos,w)$.

**Check:** the row of `fix` is `DF 246, Acc -230, AccR 0.061, PR 0.036, OddR 0.034, OddN 28152, F1 0.004`.

In [ ]:
# TODO 1 (24 pts): DF, Acc, AccR, PR, OddR, OddN, F1 for every term

def metrics_1_to_7(C):
    N_pos, N_neg, N_pos_w, N_neg_w = C['N_pos'], C['N_neg'], C['N_pos_w'], C['N_neg_w']
    N_pos_nw = N_pos - N_pos_w                     # N(Pos, not w)
    N_neg_nw = N_neg - N_neg_w                     # N(Neg, not w)
    with np.errstate(divide='ignore', invalid='ignore'):
        DF = N_pos_w + N_neg_w
        Acc = N_pos_w - N_neg_w
        AccR = ________
        PR = ________
        den1 = ________          # the slide's rule, applied to N(Neg, w)
        den2 = ________        # ... and to N(Pos, not w)
        OddR = ________
        OddN = ________
        F1 = ________
    return pd.DataFrame({'DF': DF, 'Acc': Acc, 'AccR': AccR, 'PR': PR, 'OddR': OddR, 'OddN': OddN, 'F1': F1})


m17 = metrics_1_to_7(C)
m17.index = terms
m17.loc[['fix'] + MY['words']].round(3)

### TODO 2: Metrics 8-10 (28 points)

Complete `metrics_8_to_10(C)`, which returns the columns `IG, chi2, BNS` ($Entropy_{absent}$ is given):

$$IG = Entropy_{absent} - Entropy_{given}, \qquad Entropy_{absent} = \sum_C -P(C)\log P(C),$$
$$Entropy_{given} = P(w)\sum_C -P(C|w)\log P(C|w) \;+\; P(\bar w)\sum_C -P(C|\bar w)\log P(C|\bar w)$$

$$\chi^2 = \frac{N\,\big(P(Pos,w)P(Neg,\bar w) - P(Neg,w)P(Pos,\bar w)\big)^2}{P(w)P(\bar w)P(Pos)P(Neg)} \qquad
BNS = \left|F^{-1}\!\left(\frac{N(Pos,w)}{N(Pos)}\right) - F^{-1}\!\left(\frac{N(Neg,w)}{N(Neg)}\right)\right|$$

* The logarithm is the one of the slide's worked example ($-0.6\log 0.6 - 0.4\log 0.4 = 0.29$): **base 10**, `np.log10`.
  `entropy_terms(p)` returns $-p \log p$ elementwise with $0 \log 0 = 0$ (the slide: *"Convert log(0) to zero"*).
* $\chi^2$: the full $2 \times 2$ formula above, with probabilities = counts / $N$. (scikit-learn's `chi2` is a
  different statistic and gives different numbers.)
* $BNS$: $F^{-1}$ is `norm.ppf`. Replace a rate of 1 by **0.9995** and a rate of 0 by **0.0005** (`np.clip`), and take the
  absolute value, so that a negative-class word scores as high as a positive-class word.

**Check:** the row of `fix` is `IG 0.008, chi2 204.556, BNS 1.306`.

In [ ]:
# TODO 2 (28 pts): IG, chi-squared, BNS for every term

def entropy_terms(p):
    p = np.asarray(p, dtype=float)
    out = np.zeros_like(p)
    m = p > 0                                    # 0 log 0 = 0
    out[m] = ________
    return out


def metrics_8_to_10(C):
    N_pos, N_neg, N_pos_w, N_neg_w = C['N_pos'], C['N_neg'], C['N_pos_w'], C['N_neg_w']
    N = N_pos + N_neg
    N_w = N_pos_w + N_neg_w
    N_pos_nw, N_neg_nw = N_pos - N_pos_w, N_neg - N_neg_w
    P_w, P_nw = N_w / N, 1 - N_w / N
    with np.errstate(divide='ignore', invalid='ignore'):
        E_absent = entropy_terms(N_pos / N) + entropy_terms(N_neg / N)
        E_given = (P_w * ________
                   + P_nw * ________)
        IG = ________
        chi2 = ________ \
               / ________
        lo, hi = ________, ________                 # what replaces a rate of 0 and a rate of 1
        tpr = np.clip(N_pos_w / N_pos, lo, hi)
        fpr = np.clip(N_neg_w / N_neg, lo, hi)
        BNS = ________
    return pd.DataFrame({'IG': IG, 'chi2': chi2, 'BNS': BNS})


m810 = metrics_8_to_10(C)
m810.index = terms
m810.loc[['fix'] + MY['words']].round(3)

**Self-check on the lecture's data (given).** The cell below runs your two functions on the 10 x 10 matrix of the lecture
and compares a few values with the slides. If an assertion fails, the message names the metric to fix.
(Term 3 occurs in every document, so its $\chi^2$ is $0/0$ = `NaN` here; the slide writes 0. No tweet word occurs in every
tweet, so this never happens in Part 1.)

In [ ]:
A_lecture = np.array([
    [1, 1, 1, 1, 1, 1, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 1, 1, 0, 0, 0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 1]], dtype=float)                       # terms x documents
y_lecture = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0])
X_l = A_lecture.T
C_l = dict(N_pos=6.0, N_neg=4.0, N_pos_w=X_l[y_lecture == 1].sum(0), N_neg_w=X_l[y_lecture == 0].sum(0))
L = pd.concat([metrics_1_to_7(C_l), metrics_8_to_10(C_l)], axis=1)
L.index = [f'Term {i}' for i in range(1, 11)]

expected = {('Term 3', 'DF'): 10, ('Term 2', 'Acc'): -4, ('Term 10', 'AccR'): 0.33, ('Term 9', 'PR'): 2.0,
            ('Term 1', 'OddR'): 24.0, ('Term 9', 'OddR'): 3.0, ('Term 6', 'OddN'): 12, ('Term 3', 'F1'): 0.75,
            ('Term 1', 'IG'): 0.29, ('Term 4', 'IG'): 0.10, ('Term 4', 'chi2'): 3.75, ('Term 1', 'BNS'): 6.58,
            ('Term 4', 'BNS'): 3.29}
for (t, m), v in expected.items():
    assert abs(L.loc[t, m] - v) < 0.006, f'{m} of {t}: got {L.loc[t, m]:.3f}, the slide says {v}'
print('all 13 values match the lecture slides')
L.round(2)

## &nbsp;2. Latent Semantic Analysis (32 pts)

### TODO 3: SVD with NumPy - rank and truncated reconstruction (16 points)

`entity_matrix` (given) builds a small **terms x documents** binary matrix `A` from the tweets about **your** entity
`MY['entity']` (words in at least 10 of its tweets). Decompose it with NumPy, as on the lecture's SVD slides:

1. `U, s, Vt = np.linalg.svd(A)` (full SVD; `s` holds the singular values in decreasing order). The check $U^\top U = I$ is given.
3. The **rank** of `A` is the number of **positive** singular values. Count a singular value as positive when it is larger
   than `1e-10` (smaller values are rounding noise, not dimensions).
4. Build the truncated reconstruction $A_k = U_k \Sigma_k V_k^\top$ for **your** `k = MY['k_svd']`, and report the error
   $\lVert A - A_k \rVert_F$ (`np.linalg.norm`) and the share of $\sum \sigma^2$ kept by the first $k$ singular values.

**Check:** `U^T U = I : True` for every entity. Your shapes, rank and error are not shown.

In [ ]:
def entity_matrix(entity, min_df=10):
    rows = np.where(df['Entity'].to_numpy() == entity)[0]
    sub = X[rows]
    keep = np.where(np.asarray(sub.sum(axis=0)).ravel() >= min_df)[0]
    return sub[:, keep].toarray().T.astype(float), terms[keep]       # terms x documents


A, A_terms = entity_matrix(MY['entity'])
print(MY['entity'], ': A is', A.shape, '(terms x documents)')

In [ ]:
# TODO 3 (16 pts): full SVD, rank, truncated reconstruction

U, s, Vt = ________
print('U:', U.shape, ' s:', s.shape, ' Vt:', Vt.shape)
print('U^T U = I :', np.allclose(U.T @ U, np.eye(U.shape[0])))

rank = ________
print('rank of A :', rank, '  (min(m, n) =', min(A.shape), ')')

k = MY['k_svd']
A_k = ________
err = ________
kept = ________
print(f'k = {k}: ||A - A_k||_F = {err:.3f}   share of sum(sigma^2) kept = {kept:.3f}')

# the same share for k = 1 ... 8 (given)
print('\nshare of sum(sigma^2) kept, k = 1..8:', (np.cumsum(s[:8] ** 2) / (s ** 2).sum()).round(3))

### TODO 4: LSA with `TruncatedSVD` (16 points)

Run LSA on the TF-IDF matrix `X_tfidf` (given in Section 0) with **50** latent dimensions and the **`arpack`** solver
(`random_state=7`): the grader's reference implementation uses exactly these settings, and the two solvers give slightly
different numbers (the randomized one is also reproducible with a fixed seed, but it is not the reference).

* `D` is the documents-by-50 matrix returned by `fit_transform` (the lecture's $U_k^\top A$, transposed).
* For **your** dimension `MY['dim']` (counted from 1, so the row index is `MY['dim'] - 1`), print the 5 words with the
  largest positive weight and the 5 with the largest negative weight, as on the lecture's *SVD in text mining* slide.

**Check:** `D: (7229, 50)   variance kept: 20.1%`. Your dimension's words are not shown.

In [ ]:
# TODO 4 (16 pts): TruncatedSVD, and what one dimension means

svd = TruncatedSVD(n_components=50, algorithm=________, random_state=7)
D = ________
print('D:', D.shape, f'  variance kept: {svd.explained_variance_ratio_.sum():.1%}')

dim = MY['dim'] - 1
w = svd.components_[dim]
pos_words = ________
neg_words = ________
print(f"dimension {MY['dim']}   + {list(pos_words)}")
print(f"{'':<14} - {list(neg_words)}")

## &nbsp;3. Stochastic Neighbor Embedding (16 pts)

### TODO 5: $p_{j|i}$, perplexity, and the radius $\sigma_i$ (16 points)

In the 50-dimensional LSA space `D`, implement the lecture's three steps for **one** point $i$:

$$p_{j|i} = \frac{\exp(-\lVert x_i - x_j \rVert^2 / 2\sigma_i^2)}{\sum_{k \neq i}\exp(-\lVert x_i - x_k \rVert^2 / 2\sigma_i^2)}, \qquad
Perp(P_i) = 2^{H(P_i)}, \quad H(P_i) = -\sum_j p_{j|i} \log_2 p_{j|i}$$

* `dist2` holds the **squared** Euclidean distances between all rows of `D` (`euclidean_distances(..., squared=True)`).
* `p_cond(i, sigma)`: the vector $p_{j|i}$ over all $j$, with $p_{i|i} = 0$ (set it to 0 **before** normalizing).
* `perplexity(p)`: use $\log_2$ and skip the zero entries.
* `find_sigma(i, target)`: binary search on $\sigma$ between `1e-4` and `10`: while `hi - lo > tol`, take the midpoint;
  if its perplexity is **larger** than the target, the radius is too big (move `hi`), otherwise move `lo`.
  Return the midpoint of the final interval. Use `tol = 1e-5`, so that the answer is stable to 3 decimals.

Report $\sigma$ for the check point (`i = 0`, perplexity 30) and for **your** point `MY['point']` at **your** perplexity `MY['perp']`.

**Check:** `point 0  perplexity 30 : sigma = 0.094`. Your sigma is not shown.

In [ ]:
# TODO 5 (16 pts): neighbor probabilities, perplexity, and the radius that gives a chosen perplexity

dist2 = ________


def p_cond(i, sigma):
    p = ________
    p[i] = 0
    return ________


def perplexity(p):
    nz = p[p > 0]
    return ________


def find_sigma(i, target, tol=1e-5):
    lo, hi = 1e-4, 10.0
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if ________:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2


for i, perp in [(0, 30), (MY['point'], MY['perp'])]:
    sigma = find_sigma(i, perp)
    print(f'point {i:>5}  perplexity {perp:>2} : sigma = {sigma:.3f}   (check: perplexity(p) = {perplexity(p_cond(i, sigma)):.2f})')

---
### ✅ Before you submit

- [ ] `Runtime ▸ Restart and run all` finishes with no error
- [ ] Every `________` is filled in and every output cell is visible
- [ ] Every **Check** line of the assignment matches your output, and the self-check cell prints `all 13 values match`
- [ ] `STUDENT_ID` is **your own**
- [ ] The file is named `Assignment_4_YourName_StudentID.ipynb`